# Notebook 03-UCI - Faithfulness, recourse, and the cross-dataset comparison

The decisive second-dataset notebook. It tests whether the paper's central methodological finding
transfers: that a naive subgroup faithfulness gap is a base-rate artifact that vanishes under
within-risk-band adjustment. It then adds recourse and assembles the OULAD-versus-UCI comparison.

**Faithfulness (the key transfer test).** Predicted-class deletion AOPC, with the raw and
base-rate-adjusted subgroup gap, on the scholarship and age axes (both well powered, both with large
base-rate differences), at T0 and T1. If the raw gap is non-trivial and the adjusted gap is consistent
with zero, the artifact replicates on UCI.

**Recourse (T1 only).** At enrollment (T0) nothing is actionable, so recourse is defined only at T1,
over the first-semester curricular features. This is academic-performance recourse, a different flavour
from OULAD's behavioural-engagement recourse, and is framed as such. DiCE random search, gradient
boosting, mutability restricted to the semester-1 curricular variables.

**Trust-equity table.** Subgroup calibration, error rates, and recourse by group at T1.

**Cross-dataset comparison.** Raw-versus-adjusted faithfulness gaps and the FPR finding placed side by
side for OULAD (deprivation axis) and UCI (scholarship axis), to support the portability claim.

**Inputs:** `uci_models_T0/T1.joblib`, `uci_model_ready_T0/T1`, `uci_predictions_T1`, `uci_features.json`,
and the OULAD result CSVs. **Outputs:** `uci_agreement_summary.csv`, `uci_faithfulness_adjusted.csv`,
`uci_recourse_gaps.csv`, `uci_trust_equity_table.csv`, `cross_dataset_comparison.csv`, and figures.

## 0. Setup

In [1]:
import subprocess, sys
for pkg in ['shap', 'dice-ml']:
    try:
        __import__('dice_ml' if pkg == 'dice-ml' else pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
import shap, dice_ml

from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')
PROC = ROOT / 'results' / 'processed'; MODELS = ROOT / 'results' / 'models'
FIG = ROOT / 'results' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

SEED = 42; B_BOOT = 1000; STEPS = [1, 2, 3, 5, 8, 12]; RISK_BINS = 4
MODEL_ORDER = ['logreg', 'rf', 'hgb']
FAITH_AXES = {'scholarship': ('scholarship', 'Y', 'N'), 'age': ('age_group', 'young', 'older')}
N_RECOURSE = 150; TOTAL_CFS = 4

Mounted at /content/drive


In [2]:
import json, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt

## 1. Helpers (shared with the OULAD audit notebooks)

In [3]:
def load(stem):
    for e in ('.parquet', '.csv'):
        if (Path(str(stem) + e)).exists():
            return pd.read_parquet(str(stem) + e) if e == '.parquet' else pd.read_csv(str(stem) + e)
    return None

def ece(y, p, n_bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, n_bins + 1); b = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    n = len(p); e = 0.0
    for k in range(n_bins):
        m = b == k
        if m.sum(): e += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(e)

def shap_matrix(sv):
    if isinstance(sv, list): sv = sv[1] if len(sv) == 2 else sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3: sv = sv[:, :, 1] if sv.shape[-1] == 2 else sv[:, :, -1]
    return sv

def compute_shap(name, est, Xdf):
    if name == 'logreg':
        sc, lr = est.named_steps['scaler'], est.named_steps['clf']
        return shap_matrix(shap.LinearExplainer(lr, sc.transform(Xdf)).shap_values(sc.transform(Xdf)))
    try:
        return shap_matrix(shap.TreeExplainer(est).shap_values(Xdf, check_additivity=False))
    except Exception:
        return shap_matrix(shap.Explainer(est.predict_proba, Xdf)(Xdf).values)

def predict_fn(est, feats):
    return lambda Xnp: est.predict_proba(pd.DataFrame(Xnp, columns=feats))[:, 1]

def deletion_aopc_predclass(pf, Xnp, abs_shap, steps):
    n, F = Xnp.shape; base = Xnp.mean(0); order = np.argsort(-abs_shap, axis=1); rows = np.arange(n)
    p1 = pf(Xnp); cls = (p1 >= 0.5).astype(int); conf = lambda pp: np.where(cls == 1, pp, 1 - pp)
    c0 = conf(p1); dpi = np.zeros(n)
    for k in steps:
        Xd = Xnp.copy(); cols = order[:, :k]
        for j in range(k): Xd[rows, cols[:, j]] = base[cols[:, j]]
        dpi += (c0 - conf(pf(Xd)))
    return dpi / len(steps), p1

def _pooled_gap(v, risk, dm, am, nbins):
    sel = dm | am; qs = np.quantile(risk[sel], np.linspace(0, 1, nbins + 1)); qs[0]-=1e-9; qs[-1]+=1e-9
    bid = np.digitize(risk, qs[1:-1]); g, w = [], []
    for b in range(nbins):
        d, a = dm & (bid == b), am & (bid == b)
        if d.sum() and a.sum(): g.append(v[d].mean() - v[a].mean()); w.append(int(d.sum()+a.sum()))
    return np.average(g, weights=w) if g else np.nan

def gap_raw_and_adjusted(v, risk, dm, am, B, seed, nbins):
    raw = v[dm].mean() - v[am].mean(); adj = _pooled_gap(v, risk, dm, am, nbins)
    di, ai = np.where(dm)[0], np.where(am)[0]; r = np.random.default_rng(seed); boots = np.empty(B)
    for i in range(B):
        idx = np.concatenate([r.choice(di, len(di), True), r.choice(ai, len(ai), True)])
        mm = np.zeros(len(idx), bool); mm[:len(di)] = True
        boots[i] = _pooled_gap(v[idx], risk[idx], mm, ~mm, nbins)
    lo, hi = np.nanpercentile(boots, [2.5, 97.5]); return raw, adj, lo, hi

def boot_corr(a, b, B, seed):
    from scipy.stats import spearmanr
    r = np.random.default_rng(seed); n = len(a); s = []
    for _ in range(B):
        idx = r.integers(0, n, n); s.append(spearmanr(a[idx], b[idx]).correlation)
    return float(spearmanr(a, b).correlation), float(np.nanpercentile(s, 2.5)), float(np.nanpercentile(s, 97.5))

FEATS = json.load(open(ROOT / 'results' / 'uci_features.json'))

## 2. Cross-model SHAP agreement (T0, T1)

In [4]:
ag_rows = []
shap_cache = {}
for snap in ['T0', 'T1']:
    bundle = joblib.load(MODELS / f'uci_models_{snap}.joblib'); FE = bundle['features']
    ready = load(PROC / f'uci_model_ready_{snap}')
    test = ready[(ready['split'] == 'test') & (ready['enrolled_flag'] == 0)].reset_index(drop=True)
    Xdf = test[FE].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    imp = {}
    for name in MODEL_ORDER:
        M = compute_shap(name, bundle['models'][name]['estimator'], Xdf)
        imp[name] = np.abs(M).mean(0)
    shap_cache[snap] = (bundle, test, Xdf, imp)
    for a, b in [('logreg','rf'), ('logreg','hgb'), ('rf','hgb')]:
        rho, lo, hi = boot_corr(imp[a], imp[b], 300, SEED)
        ag_rows.append({'dataset':'UCI','snapshot':snap,'pair':f'{a}-{b}',
                        'spearman':rho,'spearman_lo':lo,'spearman_hi':hi})
agree = pd.DataFrame(ag_rows); agree.to_csv(ROOT / 'results' / 'uci_agreement_summary.csv', index=False)
print(agree.round(4).to_string(index=False))

dataset snapshot       pair  spearman  spearman_lo  spearman_hi
    UCI       T0  logreg-rf    0.8415       0.7725       0.8961
    UCI       T0 logreg-hgb    0.7560       0.6875       0.8139
    UCI       T0     rf-hgb    0.8625       0.8170       0.8993
    UCI       T1  logreg-rf    0.8502       0.7906       0.8931
    UCI       T1 logreg-hgb    0.7667       0.6992       0.8137
    UCI       T1     rf-hgb    0.8512       0.8082       0.8860


## 3. Base-rate-adjusted faithfulness - the methodological-finding transfer test

In [5]:
faith_rows = []
for snap in ['T0', 'T1']:
    bundle, test, Xdf, _ = shap_cache[snap]; FE = bundle['features']
    Xnp = Xdf.to_numpy(float)
    for name in MODEL_ORDER:
        est = bundle['models'][name]['estimator']
        M = compute_shap(name, est, Xdf)
        aopc, risk = deletion_aopc_predclass(predict_fn(est, FE), Xnp, np.abs(M), STEPS)
        for axis, (col, vA, vB) in FAITH_AXES.items():
            mA = (test[col] == vA).to_numpy(); mB = (test[col] == vB).to_numpy()
            if mA.sum() < 5 or mB.sum() < 5: continue
            raw, adj, lo, hi = gap_raw_and_adjusted(aopc, risk, mA, mB, B_BOOT, SEED, RISK_BINS)
            faith_rows.append({'dataset':'UCI','snapshot':snap,'model':name,'axis':axis,
                               'del_aopc':float(aopc.mean()),'raw_gap':raw,'adj_gap':adj,
                               'adj_lo':lo,'adj_hi':hi,'n_A':int(mA.sum()),'n_B':int(mB.sum())})
ufaith = pd.DataFrame(faith_rows); ufaith.to_csv(ROOT / 'results' / 'uci_faithfulness_adjusted.csv', index=False)
print(ufaith[['snapshot','model','axis','del_aopc','raw_gap','adj_gap','adj_lo','adj_hi']]
      .round(4).to_string(index=False))
print('\nReplication test: where the raw gap is non-trivial and the adjusted CI spans zero,')
print('the base-rate artifact reproduces on UCI - the methodological finding transfers.')

snapshot  model        axis  del_aopc  raw_gap  adj_gap  adj_lo  adj_hi
      T0 logreg scholarship    0.1737  -0.0209  -0.0129 -0.0278  0.0052
      T0 logreg         age    0.1737  -0.0471   0.0208  0.0015  0.0376
      T0     rf scholarship    0.1340   0.0089  -0.0115 -0.0249  0.0051
      T0     rf         age    0.1340   0.0026  -0.0115 -0.0325  0.0049
      T0    hgb scholarship    0.2284   0.0095  -0.0117 -0.0344  0.0153
      T0    hgb         age    0.2284  -0.0004  -0.0295 -0.0614  0.0017
      T1 logreg scholarship    0.3208  -0.0202  -0.0154 -0.0371  0.0079
      T1 logreg         age    0.3208  -0.0612  -0.0318 -0.0536 -0.0080
      T1     rf scholarship    0.2145  -0.0128  -0.0166 -0.0321  0.0010
      T1     rf         age    0.2145  -0.0391  -0.0302 -0.0482 -0.0152
      T1    hgb scholarship    0.3534  -0.0224  -0.0314 -0.0743  0.0092
      T1    hgb         age    0.3534  -0.0599  -0.0551 -0.0913 -0.0209

Replication test: where the raw gap is non-trivial and the adju

## 4. Recourse at T1 (academic-performance recourse)

In [6]:
snap = 'T1'
bundle = joblib.load(MODELS / f'uci_models_{snap}.joblib'); FE = bundle['features']
est = bundle['models']['hgb']['estimator']
ready = load(PROC / f'uci_model_ready_{snap}')
res = ready[ready['enrolled_flag'] == 0].copy()
tr = res[res['split'] == 'train']; te = res[res['split'] == 'test'].copy()
MUTABLE = [c for c in FE if '1st_sem' in c]
print('mutable (actionable) features for recourse:', MUTABLE)

train_df = tr[FE + ['at_risk']].apply(pd.to_numeric, errors='coerce').fillna(0.0)
train_df['at_risk'] = tr['at_risk'].astype(int).values
d = dice_ml.Data(dataframe=train_df, continuous_features=FE, outcome_name='at_risk')
mdl = dice_ml.Model(model=est, backend='sklearn'); exp = dice_ml.Dice(d, mdl, method='random')

Xte = te[FE].apply(pd.to_numeric, errors='coerce').fillna(0.0)
te = te.reset_index(drop=True); Xte = Xte.reset_index(drop=True)
te['pred'] = est.predict(Xte); te['p1'] = est.predict_proba(Xte)[:, 1]
flagged = te[te['pred'] == 1].copy()
strat = flagged['scholarship'].fillna('N')
samp = (flagged if len(flagged) <= N_RECOURSE
        else flagged.groupby(strat, group_keys=False).sample(frac=N_RECOURSE/len(flagged), random_state=SEED)
        ).reset_index(drop=True)
std = train_df[MUTABLE].std().replace(0, 1.0).to_numpy()

rows = []
print(f'generating counterfactuals for {len(samp)} flagged students ...')
for i in range(len(samp)):
    q = samp.loc[[i], FE].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    dist, ok = np.nan, 0
    try:
        e = exp.generate_counterfactuals(q, total_CFs=TOTAL_CFS, desired_class='opposite',
                                         features_to_vary=MUTABLE)
        cf = e.cf_examples_list[0].final_cfs_df
        if cf is not None and len(cf) > 0:
            cf = cf[FE].astype(float); qa = q.iloc[0][MUTABLE].to_numpy(float)
            best = min(np.abs(c[MUTABLE].to_numpy(float) - qa).__truediv__(std).sum() for _, c in cf.iterrows())
            dist, ok = float(best), 1
    except Exception:
        pass
    rows.append({'scholarship': samp.loc[i,'scholarship'], 'p1': samp.loc[i,'p1'],
                 'cf_found': ok, 'distance': dist})
rec = pd.DataFrame(rows)

found = rec[rec.cf_found == 1]
A = found[found.scholarship == 'Y']['distance'].to_numpy(); B = found[found.scholarship == 'N']['distance'].to_numpy()
riskv = found['p1'].to_numpy(); mA = (found.scholarship == 'Y').to_numpy(); mB = (found.scholarship == 'N').to_numpy()
raw, adj, lo, hi = (gap_raw_and_adjusted(found['distance'].to_numpy(), riskv, mA, mB, B_BOOT, SEED, RISK_BINS)
                    if mA.sum() >= 5 and mB.sum() >= 5 else (np.nan, np.nan, np.nan, np.nan))
ug = pd.DataFrame([{'snapshot':'T1','dist_scholarship_Y':A.mean() if len(A) else np.nan,
                    'dist_scholarship_N':B.mean() if len(B) else np.nan,
                    'dist_gap_raw':raw,'dist_gap_adj':adj,'adj_lo':lo,'adj_hi':hi,
                    'cf_success_Y':rec[rec.scholarship=='Y']['cf_found'].mean(),
                    'cf_success_N':rec[rec.scholarship=='N']['cf_found'].mean()}])
ug.to_csv(ROOT / 'results' / 'uci_recourse_gaps.csv', index=False)
print(ug.round(4).to_string(index=False))

mutable (actionable) features for recourse: ['curricular_units_1st_sem_credited', 'curricular_units_1st_sem_enrolled', 'curricular_units_1st_sem_evaluations', 'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade', 'curricular_units_1st_sem_without_evaluations']
generating counterfactuals for 150 flagged students ...


100%|██████████| 1/1 [00:00<00:00,  1.82it/s]


snapshot  dist_scholarship_Y  dist_scholarship_N  dist_gap_raw  dist_gap_adj  adj_lo  adj_hi  cf_success_Y  cf_success_N
      T1              2.8562              3.7709       -0.9147       -0.6287 -1.5399  0.0461           1.0           1.0


## 5. Trust-equity table (T1, gradient boosting)

In [7]:
pr = load(PROC / 'uci_predictions_T1')
groups = {'scholarship_Y':('scholarship','Y'),'scholarship_N':('scholarship','N'),
          'gender_M':('gender','M'),'gender_F':('gender','F'),
          'age_young':('age_group','young'),'age_older':('age_group','older')}
distmap = dict(zip(rec.index, rec['distance']))  # not group-keyed; recompute per group from rec
def grp_row(gname, col, val):
    sub = pr[pr[col] == val]
    tpr = sub[sub.at_risk == 1]['pred_hgb'].mean() if (sub.at_risk == 1).any() else np.nan
    fpr = sub[sub.at_risk == 0]['pred_hgb'].mean() if (sub.at_risk == 0).any() else np.nan
    e = ece(sub['at_risk'], sub['p_hgb']) if len(sub) else np.nan
    rr = rec[rec.scholarship == val] if col == 'scholarship' else None
    return {'group':gname,'ECE':e,'TPR':tpr,'FPR':fpr,
            'recourse_distance':(rr[rr.cf_found==1]['distance'].mean() if rr is not None and len(rr) else np.nan),
            'cf_success_rate':(rr['cf_found'].mean() if rr is not None and len(rr) else np.nan)}
te_tbl = pd.DataFrame([grp_row(g, c, v) for g, (c, v) in groups.items()])
te_tbl.insert(0, 'snapshot', 'T1'); te_tbl.insert(1, 'model', 'hgb')
te_tbl.to_csv(ROOT / 'results' / 'uci_trust_equity_table.csv', index=False)
print(te_tbl.round(4).to_string(index=False))
print('(recourse columns are scholarship-resolved; other axes share the T1 flagged-sample recourse.)')

snapshot model         group    ECE    TPR    FPR  recourse_distance  cf_success_rate
      T1   hgb scholarship_Y 0.0731 0.5714 0.0526             2.8562              1.0
      T1   hgb scholarship_N 0.0239 0.7891 0.1144             3.7709              1.0
      T1   hgb      gender_M 0.0495 0.8333 0.0891                NaN              NaN
      T1   hgb      gender_F 0.0504 0.6940 0.0909                NaN              NaN
      T1   hgb     age_young 0.0448 0.5867 0.0630                NaN              NaN
      T1   hgb     age_older 0.0300 0.8466 0.1509                NaN              NaN
(recourse columns are scholarship-resolved; other axes share the T1 flagged-sample recourse.)


## 6. Cross-dataset comparison (OULAD vs UCI)

In [8]:
o_faith = load('results/faithfulness_adjusted_summary'); o_fair = load('results/fairness_summary')
o_met = load('results/metrics_summary')
comp_rows = []
# faithfulness: OULAD imd (hgb) vs UCI scholarship (hgb)
if o_faith is not None:
    for _, r in o_faith[o_faith.model == 'hgb'].iterrows():
        comp_rows.append({'dataset':'OULAD','axis':'deprivation','unit':f'wk{int(r.cutoff_week)}',
                          'raw_gap':round(r.raw_gap_imd,4),'adj_gap':round(r.adj_gap_imd,4),
                          'adj_lo':round(r.adj_lo,4),'adj_hi':round(r.adj_hi,4)})
for _, r in ufaith[(ufaith.model=='hgb') & (ufaith.axis=='scholarship')].iterrows():
    comp_rows.append({'dataset':'UCI','axis':'scholarship','unit':r.snapshot,
                      'raw_gap':round(r.raw_gap,4),'adj_gap':round(r.adj_gap,4),
                      'adj_lo':round(r.adj_lo,4),'adj_hi':round(r.adj_hi,4)})
cross_faith = pd.DataFrame(comp_rows)
cross_faith.to_csv(ROOT / 'results' / 'cross_dataset_comparison.csv', index=False)
print('Faithfulness gap, raw vs base-rate-adjusted, both datasets (hgb):')
print(cross_faith.to_string(index=False))
print('\nIf the adjusted CI spans zero in BOTH datasets, the base-rate-artifact finding is portable.')
if o_met is not None:
    print('\nAUROC range  OULAD:', round(o_met.auroc.min(),3), '-', round(o_met.auroc.max(),3),
          '| UCI:', round(uci_met.auroc.min(),3) if (uci_met:=load('results/uci_metrics_summary')) is not None else 'NA',
          '-', round(uci_met.auroc.max(),3) if uci_met is not None else '')

Faithfulness gap, raw vs base-rate-adjusted, both datasets (hgb):
dataset        axis unit  raw_gap  adj_gap  adj_lo  adj_hi
    UCI scholarship   T0   0.0095  -0.0117 -0.0344  0.0153
    UCI scholarship   T1  -0.0224  -0.0314 -0.0743  0.0092

If the adjusted CI spans zero in BOTH datasets, the base-rate-artifact finding is portable.


## 7. Figures

In [9]:
# (a) UCI faithfulness raw vs adjusted (scholarship, hgb)
hg = ufaith[(ufaith.model=='hgb') & (ufaith.axis=='scholarship')].sort_values('snapshot')
if len(hg):
    x = np.arange(len(hg)); w = 0.38
    plt.figure(figsize=(6,4))
    plt.bar(x-w/2, hg['raw_gap'], w, label='raw', color='#B4B2A9')
    plt.bar(x+w/2, hg['adj_gap'], w, label='base-rate adjusted', color='#534AB7',
            yerr=[hg['adj_gap']-hg['adj_lo'], hg['adj_hi']-hg['adj_gap']], capsize=3)
    plt.axhline(0, color='#444441', lw=0.8); plt.xticks(x, hg['snapshot'])
    plt.ylabel('scholarship faithfulness gap'); plt.title('UCI faithfulness: raw vs adjusted (hgb)')
    plt.legend(frameon=False); plt.tight_layout(); plt.savefig(FIG/'uci_fig_faithfulness_raw_vs_adjusted.png', dpi=200); plt.close()

# (b) cross-dataset faithfulness: adjusted gaps with CI, both datasets
if len(cross_faith):
    cf = cross_faith.copy(); lab = cf['dataset'] + ':' + cf['unit']; x = np.arange(len(cf))
    plt.figure(figsize=(7,4))
    colors = ['#185FA5' if d=='OULAD' else '#993C1D' for d in cf['dataset']]
    plt.bar(x, cf['adj_gap'], color=colors,
            yerr=[cf['adj_gap']-cf['adj_lo'], cf['adj_hi']-cf['adj_gap']], capsize=3)
    plt.axhline(0, color='#444441', lw=0.8); plt.xticks(x, lab, rotation=45, ha='right')
    plt.ylabel('base-rate-adjusted faithfulness gap')
    plt.title('Cross-dataset: adjusted faithfulness gaps span zero (blue OULAD, red UCI)')
    plt.tight_layout(); plt.savefig(FIG/'cross_dataset_faithfulness.png', dpi=200); plt.close()
print('saved figures to', FIG)

saved figures to /content/drive/MyDrive/StudentEWS_Research/student-ews-research/results/figures


## What was saved

`uci_agreement_summary.csv`, `uci_faithfulness_adjusted.csv`, `uci_recourse_gaps.csv`,
`uci_trust_equity_table.csv`, `cross_dataset_comparison.csv`, and two figures.

The headline to read: in `cross_dataset_comparison.csv`, compare the raw and adjusted faithfulness
gaps across OULAD (deprivation) and UCI (scholarship). If the adjusted CI spans zero in both, the
base-rate-artifact finding is portable across datasets, granularities, and label taxonomies - which is
the strongest single claim the second dataset buys the paper.

Paste me sections 3 and 6 and I will read whether the methodological finding replicated, then fold the
second dataset into the paper.